# TP2 — Classificação de Imagens com Aprendizado de Máquina Clássico

**Disciplina:** Aprendizado de Máquina I — Prof. Cristiano Rodrigues — PUC Minas
**Integrante:** Pedro Dias Soares - 879672

---

### Sobre o trabalho
A ideia aqui é aplicar algoritmos clássicos de machine learning num problema de classificação de imagens. No
fundo, uma imagem é só uma matriz de números que a gente transforma num vetor de atributos, e é em cima desse
vetor que os modelos trabalham. Montei um pipeline supervisionado completo, comparei alguns modelos e, no fim,
analisei os resultados.

### Como o notebook está organizado
1. Configuração e reprodutibilidade (imports, versões, `random_state`).
2. Funções utilitárias, que reaproveito nas duas partes para não repetir código.
3. Parte A — Digits (obrigatória): o pipeline completo.
4. Parte B — CIFAR-10: o mesmo pipeline, com o carregamento adaptado para o Kaggle.
5. Discussão e conclusão.

### Decisões principais
- Dataset da Parte B: CIFAR-10 (10 classes, 32×32, convertido para tons de cinza).
- Três modelos: KNN, Random Forest e Naive Bayes Gaussiano — peguei um de cada "família" (vizinhança,
  ensemble e probabilístico).
- Duas formas de representar a imagem: os pixels brutos e um conjunto de atributos derivados (estatísticas por
  região + histograma).

## 1. Configuração e reprodutibilidade

Importo só as bibliotecas que o enunciado permite (numpy, pandas, matplotlib, seaborn e scikit-learn) e fixo
uma semente única (`SEED`) para o resultado não mudar a cada execução. Como o trabalho cobra reprodutibilidade
— fixar `random_state`, registrar as versões e garantir que o notebook rode do começo ao fim —, deixo isso
resolvido logo de início.

In [ ]:
import sys, time, glob, warnings          # utilitarios: versao do python, cronometro, busca de arquivos e supressao de avisos
import numpy as np                          # numpy: toda a manipulacao numerica (vetores e matrizes das imagens)
import pandas as pd                         # pandas: tabelas de resultados (DataFrame)
import matplotlib.pyplot as plt             # matplotlib: base dos graficos
import seaborn as sns                       # seaborn: graficos mais bonitos (barras e heatmap da matriz de confusao)
import sklearn                              # scikit-learn: importo so para imprimir a versao

from sklearn.datasets import load_digits, fetch_openml                       # carregadores de datasets prontos
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV  # split, validacao cruzada e busca de hiperparametros
from sklearn.preprocessing import StandardScaler                             # padronizacao das features (media 0, desvio 1)
from sklearn.pipeline import Pipeline                                        # encadeia scaler + modelo para evitar vazamento
from sklearn.neighbors import KNeighborsClassifier                          # modelo 1: KNN (vizinhanca)
from sklearn.ensemble import RandomForestClassifier                        # modelo 2: Random Forest (ensemble de arvores)
from sklearn.naive_bayes import GaussianNB                                   # modelo 3: Naive Bayes Gaussiano (probabilistico)
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,     # metricas de avaliacao
                             classification_report)                         # relatorio com precision/recall/f1 por classe

warnings.filterwarnings("ignore")           # esconde avisos para a saida ficar limpa
sns.set_theme(style="whitegrid")            # define o estilo visual padrao dos graficos

# Semente global de reprodutibilidade
SEED = 42                                   # uma unica semente usada em todo o notebook
np.random.seed(SEED)                        # fixa o gerador aleatorio do numpy -> resultados nao mudam a cada execucao

# versoes das libs, so pra registrar o ambiente
print("Python     :", sys.version.split()[0])  # imprime a versao do Python
print("numpy      :", np.__version__)           # versao do numpy
print("pandas     :", pd.__version__)           # versao do pandas
print("scikit-learn:", sklearn.__version__)     # versao do scikit-learn
print("seaborn    :", sns.__version__)          # versao do seaborn
print("\nSEED =", SEED)                          # confirma a semente usada

## 2. Funções utilitárias

Para não repetir código nas duas partes, juntei aqui as funções que uso tanto no Digits quanto no CIFAR-10.
Cada uma faz uma coisa só (carregar e documentar, extrair atributos, treinar, avaliar, plotar) e elas vão
sendo chamadas mais à frente.

### 2.1 Documentação e exploração do dataset

Duas funções: uma imprime os metadados do dataset (quantas amostras, classes, resolução, canais e a
distribuição) e a outra mostra um exemplo de cada classe.

In [ ]:
def document_dataset(name, X, y, img_side, channels, class_names):
    '''Imprime os metadados basicos do dataset.'''
    print(f"=== {name} ===")                                  # cabecalho com o nome do dataset
    print("Numero de amostras      :", X.shape[0])            # X.shape[0] = quantidade de imagens (linhas)
    print("Atributos (pixels)      :", X.shape[1])            # X.shape[1] = quantidade de atributos/pixels (colunas)
    print(f"Resolucao               : {img_side}x{img_side}") # resolucao da imagem (lado x lado)
    print("Numero de canais        :", channels, "(tons de cinza)")  # 1 canal = imagem em cinza
    print("Numero de classes       :", len(np.unique(y)))     # np.unique(y) = rotulos distintos -> numero de classes
    dist = pd.Series(y).value_counts().sort_index()           # conta quantas amostras ha em cada classe, ordenado pelo rotulo
    dist.index = [class_names[i] for i in dist.index]         # troca o numero da classe pelo nome legivel
    print("\nDistribuicao de classes:")                       # titulo da distribuicao
    print(dist.to_string())                                   # imprime a contagem por classe


def plot_class_examples(X, y, img_side, class_names, title):
    '''Mostra um exemplo de cada classe.'''
    classes = np.unique(y)                                    # lista das classes existentes
    fig, axes = plt.subplots(1, len(classes), figsize=(1.5 * len(classes), 2.2))  # uma coluna por classe
    for ax, c in zip(np.ravel(axes), classes):               # percorre cada subplot junto com a classe correspondente
        idx = np.where(y == c)[0][0]                          # pega o indice da PRIMEIRA imagem daquela classe
        ax.imshow(X[idx].reshape(img_side, img_side), cmap="gray")  # transforma o vetor de volta em imagem 2D e desenha em cinza
        ax.set_title(str(class_names[c]), fontsize=8)         # titulo do subplot = nome da classe
        ax.axis("off")                                        # esconde os eixos (numeros) para so mostrar a imagem
    fig.suptitle(title)                                       # titulo geral da figura
    plt.tight_layout()                                        # ajusta o espacamento para nao sobrepor
    plt.show()                                                # exibe a figura


def plot_class_distribution(y, class_names, title):
    '''Distribuicao das classes, pra checar o balanceamento.'''
    vals, counts = np.unique(y, return_counts=True)           # vals = classes, counts = quantas amostras em cada uma
    plt.figure(figsize=(8, 3))                                # tamanho do grafico
    sns.barplot(x=[str(class_names[v]) for v in vals], y=counts, color="#4C72B0")  # barra: nome da classe x quantidade
    plt.title(title)                                          # titulo do grafico
    plt.ylabel("Quantidade")                                  # rotulo do eixo y
    plt.xticks(rotation=45, ha="right")                       # inclina os nomes das classes para nao sobrepor
    plt.tight_layout()                                        # ajusta o espacamento
    plt.show()                                                # exibe o grafico

### 2.2 Engenharia de atributos — Representação 2

A Representação 1 é a imagem achatada num vetor de pixels. A Representação 2 resume cada imagem em poucos
números, e fiz tudo só com NumPy para não fugir das bibliotecas permitidas. Ela junta três coisas:

- Estatísticas por região: divido a imagem numa grade e tiro a média e o desvio de cada bloco. É como uma
  versão reduzida da imagem que ainda guarda um pouco da posição espacial.
- Histograma de intensidades: a distribuição de tons claros e escuros, que é pouco sensível a pequenos
  deslocamentos do objeto.
- Proporção de pixels claros e escuros: dois números globais bem simples.

Combinei o histograma com as estatísticas por região de propósito: sozinho, o histograma perde a posição dos
pixels, e os blocos compensam isso em parte.

In [ ]:
def extract_features(X_flat, img_side, value_max, grid=4, n_bins=16):
    '''Representacao 2: estatisticas por regiao + histograma + proporcoes claro/escuro.

    Observacao sobre vazamento de dados: a extracao e feita POR IMAGEM (nao usa
    estatisticas do conjunto), portanto pode ser calculada antes do split sem
    causar data leakage. A padronizacao (StandardScaler) e que fica DENTRO do
    Pipeline, com fit apenas no treino.
    '''
    n = X_flat.shape[0]                                       # numero de imagens recebidas
    imgs = X_flat.reshape(n, img_side, img_side).astype(np.float32) / float(value_max)  # volta para 2D e normaliza para [0, 1]

    # 1) Estatisticas por regiao (grade grid x grid)
    bh = img_side // grid                     # altura de cada bloco da grade
    bw = img_side // grid                     # largura de cada bloco da grade
    usable = bh * grid                        # parte da imagem que cabe certinho na grade (recorta o resto)
    cropped = imgs[:, :usable, :usable]       # recorta a imagem para ser divisivel pela grade
    blocks = cropped.reshape(n, grid, bh, grid, bw)          # reorganiza em blocos: (imagem, linha_grade, alt_bloco, col_grade, larg_bloco)
    block_mean = blocks.mean(axis=(2, 4)).reshape(n, -1)   # media de cada bloco -> grid*grid atributos
    block_std = blocks.std(axis=(2, 4)).reshape(n, -1)     # desvio padrao de cada bloco -> grid*grid atributos

    # 2) Histograma de intensidades (normalizado para somar 1)
    hist = np.stack([np.histogram(imgs[i], bins=n_bins, range=(0.0, 1.0))[0]   # conta pixels por faixa de intensidade, imagem a imagem
                     for i in range(n)]).astype(np.float32)
    hist = hist / (hist.sum(axis=1, keepdims=True) + 1e-8)  # normaliza o histograma (some 1); +1e-8 evita divisao por zero

    # 3) Proporcao de pixels claros e escuros
    light = (imgs > 0.5).reshape(n, -1).mean(axis=1, keepdims=True)  # fracao de pixels claros (intensidade > 0.5)
    dark = (imgs < 0.1).reshape(n, -1).mean(axis=1, keepdims=True)   # fracao de pixels escuros (intensidade < 0.1)

    return np.hstack([block_mean, block_std, hist, light, dark]).astype(np.float32)  # junta tudo num unico vetor de atributos por imagem

### 2.3 Modelos e a função de experimento

Aqui ficam os três modelos e uma função única que treina, ajusta os hiperparâmetros e avalia, para todos
seguirem exatamente o mesmo protocolo. Alguns pontos que valem nota:

- Uso `Pipeline(StandardScaler + modelo)`, então a padronização é aprendida só com o treino, dentro de cada
  fold da validação cruzada. É assim que se evita o vazamento de dados.
- Só escalono KNN e Naive Bayes, que dependem da magnitude das features. A Random Forest não precisa, porque
  divide por limiares e não liga para a escala.
- A busca de hiperparâmetros é um `GridSearchCV` com `StratifiedKFold`, otimizando F1-macro.
- Também cronometro treino e inferência, que é o que o item 8 pede.

In [ ]:
def get_models(seed):
    '''(nome, estimador, grade de hiperparametros, precisa_escala).'''
    return [
        ("KNN",                                  # nome do modelo 1
         KNeighborsClassifier(),                 # estimador KNN (sem parametros fixos aqui)
         {"n_neighbors": [3, 5, 7]},             # valores de k testados na busca
         True),   # KNN usa distancia, entao precisa de escala
        ("Random Forest",                        # nome do modelo 2
         RandomForestClassifier(random_state=seed, n_jobs=-1),  # floresta com semente fixa e usando todos os nucleos
         {"n_estimators": [100, 200]},           # numero de arvores testado na busca
         False),  # arvore nao liga pra escala
        ("Naive Bayes",                          # nome do modelo 3
         GaussianNB(),                           # Naive Bayes Gaussiano
         {"var_smoothing": [1e-9, 1e-7, 1e-5]},  # suavizacao da variancia testada na busca
         True),   # NB Gaussiano: escala ajuda a estimativa de sigma
    ]


def run_experiment(name, estimator, param_grid, X_tr, X_te, y_tr, y_te,
                   need_scale, cv, seed):
    '''Treina (com CV/GridSearch), avalia no teste e mede tempos.'''
    steps = []                                   # lista de etapas do pipeline
    if need_scale:                               # se o modelo precisa de escala...
        steps.append(("scaler", StandardScaler()))  # ...adiciona a padronizacao como primeira etapa
    steps.append(("clf", estimator))             # adiciona o classificador como ultima etapa
    pipe = Pipeline(steps)                        # monta o pipeline (scaler -> modelo) ou so o modelo

    grid = {f"clf__{k}": v for k, v in param_grid.items()}   # prefixa os parametros com "clf__" (nome da etapa do pipeline)
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="f1_macro", n_jobs=-1)  # busca em grade com CV, otimizando F1-macro

    t0 = time.perf_counter(); gs.fit(X_tr, y_tr); t_train = time.perf_counter() - t0   # treina e cronometra o tempo de treino
    t0 = time.perf_counter(); y_pred = gs.predict(X_te); t_infer = time.perf_counter() - t0  # prediz no teste e cronometra a inferencia

    return {                                     # devolve um dicionario com tudo que interessa do experimento
        "model": name,                           # nome do modelo
        "best_params": gs.best_params_,          # melhores hiperparametros encontrados
        "cv_f1": gs.best_score_,                 # melhor F1-macro na validacao cruzada
        "test_acc": accuracy_score(y_te, y_pred),            # acuracia no conjunto de teste
        "test_f1": f1_score(y_te, y_pred, average="macro"),  # F1-macro no conjunto de teste
        "train_time_s": t_train,                 # tempo de treino em segundos
        "infer_time_s": t_infer,                 # tempo de inferencia em segundos
        "y_pred": y_pred,                        # predicoes (usadas depois na matriz de confusao)
        "estimator": gs.best_estimator_,         # o melhor modelo ja treinado
    }


def run_all_models(X_tr, X_te, y_tr, y_te, rep_label, cv, seed):
    '''Roda os 3 modelos para uma representacao e devolve a lista de resultados.'''
    rows = []                                    # acumula os resultados de cada modelo
    for name, est, grid, need_scale in get_models(seed):   # percorre os 3 modelos definidos em get_models
        print(f"  -> {name} ({rep_label}) ...", end=" ")   # mostra qual modelo esta rodando
        r = run_experiment(name, est, grid, X_tr, X_te, y_tr, y_te, need_scale, cv, seed)  # treina e avalia esse modelo
        r["representation"] = rep_label          # marca qual representacao foi usada (Rep1 ou Rep2)
        rows.append(r)                           # guarda o resultado
        print(f"acc={r['test_acc']:.3f} | F1={r['test_f1']:.3f} | "  # imprime um resumo rapido
              f"treino={r['train_time_s']:.1f}s")
    return rows                                  # devolve a lista com os 3 resultados

### 2.4 Avaliação e comparação

Funções para a avaliação (matriz de confusão e relatório com precision, recall e F1) e para as comparações
(modelos entre si, Rep1 contra Rep2 e o tempo de cada um). Gosto de olhar a matriz de confusão menos como nota
e mais como diagnóstico: ela mostra com quais classes o modelo se embola, não só o quanto ele erra.

In [ ]:
def results_to_df(rows):
    '''Converte a lista de resultados em DataFrame resumido.'''
    cols = ["model", "representation", "cv_f1", "test_acc", "test_f1",   # colunas que quero manter na tabela
            "train_time_s", "infer_time_s"]
    return pd.DataFrame([{c: r[c] for c in cols} for r in rows])         # monta um DataFrame so com essas colunas


def plot_confusions(rows, y_true, class_names, title):
    '''Plota a matriz de confusao de cada modelo lado a lado.'''
    k = len(rows)                                            # quantos modelos serao plotados
    fig, axes = plt.subplots(1, k, figsize=(5.2 * k, 4.6))  # um subplot por modelo, lado a lado
    for ax, r in zip(np.ravel(axes), rows):                 # percorre cada subplot junto com o resultado do modelo
        cm = confusion_matrix(y_true, r["y_pred"])          # calcula a matriz de confusao (real x predito)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,  # desenha a matriz como mapa de calor com numeros
                    xticklabels=class_names, yticklabels=class_names,          # rotula linhas e colunas com os nomes das classes
                    annot_kws={"size": 7})                                     # tamanho da fonte dos numeros
        ax.set_title(f"{r['model']} - {r['representation']}\n"   # titulo: modelo + representacao...
                     f"acc={r['test_acc']:.3f} | F1={r['test_f1']:.3f}", fontsize=9)  # ...e as metricas
        ax.set_xlabel("Predito"); ax.set_ylabel("Real")     # eixos: colunas = predito, linhas = real
        ax.tick_params(axis="x", rotation=45)               # inclina os rotulos do eixo x
        ax.tick_params(axis="y", rotation=0)                # mantem os rotulos do eixo y na horizontal
    fig.suptitle(title)                                     # titulo geral
    plt.tight_layout()                                      # ajusta o espacamento
    plt.show()                                              # exibe


def plot_model_comparison(df, title):
    '''Barras de F1-macro por modelo, comparando as duas representacoes.'''
    plt.figure(figsize=(8, 4))                              # tamanho do grafico
    sns.barplot(data=df, x="model", y="test_f1", hue="representation")  # barras de F1 por modelo, separadas por representacao
    plt.title(title); plt.ylabel("F1-macro (teste)"); plt.ylim(0, 1)   # titulo, rotulo do y e escala fixa de 0 a 1
    plt.legend(title="Representacao")                       # legenda identificando Rep1 e Rep2
    plt.tight_layout(); plt.show()                          # ajusta e exibe


def plot_cost_comparison(df, title):
    '''Barras de tempo de treino e inferencia (item 8: custo computacional).'''
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))         # dois graficos lado a lado: treino e inferencia
    sns.barplot(data=df, x="model", y="train_time_s", hue="representation", ax=axes[0])  # tempo de treino por modelo
    axes[0].set_title("Tempo de treino (s)"); axes[0].set_ylabel("segundos")             # titulo e rotulo do primeiro grafico
    sns.barplot(data=df, x="model", y="infer_time_s", hue="representation", ax=axes[1])  # tempo de inferencia por modelo
    axes[1].set_title("Tempo de inferencia (s)"); axes[1].set_ylabel("segundos")         # titulo e rotulo do segundo grafico
    fig.suptitle(title)                                     # titulo geral
    plt.tight_layout(); plt.show()                          # ajusta e exibe


def best_config(rows):
    '''Retorna o resultado de maior F1-macro no teste.'''
    return max(rows, key=lambda r: r["test_f1"])            # escolhe, entre os resultados, o de maior F1-macro no teste

---
# Parte A — Digits (obrigatória)

O Digits já vem pronto no scikit-learn (`load_digits`): são 1.797 imagens 8×8 em tons de cinza de dígitos
manuscritos de 0 a 9, com intensidade de 0 a 16. É um dataset pequeno e controlado, bom para validar o
pipeline antes de partir para algo mais pesado.

## A.1 Carregamento e documentação

In [ ]:
digits = load_digits()                         # carrega o dataset Digits embutido no scikit-learn
X_digits = digits.data.astype(np.float32)      # (1797, 64) - pixels achatados, convertidos para float
y_digits = digits.target.astype(int)           # rotulos 0..9 como inteiros
DIGITS_SIDE = 8                                 # cada imagem tem 8x8 pixels
DIGITS_MAX = float(X_digits.max())             # 16.0 (intensidade maxima; constante de dominio, nao e leakage)
DIGITS_NAMES = [str(i) for i in range(10)]     # nomes das classes: "0", "1", ..., "9"

document_dataset("Digits", X_digits, y_digits, DIGITS_SIDE, 1, DIGITS_NAMES)  # imprime os metadados do Digits

## A.2 Visualização exploratória

Mostro um exemplo de cada dígito e a distribuição das classes. Dá para comentar a variação dentro da mesma
classe (o mesmo dígito escrito de jeitos diferentes) e a semelhança entre algumas classes — 4 e 9, ou 1 e 8,
têm traços parecidos e costumam se confundir.

In [ ]:
plot_class_examples(X_digits, y_digits, DIGITS_SIDE, DIGITS_NAMES,    # mostra um exemplo de cada digito (0 a 9)
                    "Digits - um exemplo por classe")
plot_class_distribution(y_digits, DIGITS_NAMES, "Digits - distribuicao de classes")  # mostra quantas imagens ha por classe

As classes estão praticamente equilibradas (~180 imagens por dígito). Como a base é balanceada, a
acurácia já seria confiável, mas mesmo assim reporto F1-macro e a matriz de confusão para não depender só
dela.

## A.3 Divisão treino/teste

Faço um holdout estratificado 80/20 (com `stratify=y`, para manter a proporção das classes) e já defino a
validação cruzada de 5 folds que vai ser usada na busca de hiperparâmetros. O teste fica reservado e só é
usado no fim — todo o ajuste acontece dentro do treino, via CV.

In [ ]:
Xtr_d, Xte_d, ytr_d, yte_d = train_test_split(                       # divide em treino e teste
    X_digits, y_digits, test_size=0.20, stratify=y_digits, random_state=SEED)  # 80/20, estratificado e com semente fixa

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)    # validacao cruzada de 5 folds para a busca de hiperparametros

print("Treino:", Xtr_d.shape, "| Teste:", Xte_d.shape)              # confere os tamanhos do treino e do teste

## A.4 Pré-processamento

Não normalizo os pixels na mão. Deixo a padronização dentro do Pipeline de cada modelo, só onde faz sentido
(KNN e Naive Bayes), com o `fit` acontecendo apenas no treino de cada fold. Assim não tem vazamento e todos os
modelos competem em condições iguais.

## A.5 As duas representações

- Rep1: os 64 pixels da imagem 8×8 achatados.
- Rep2: estatísticas por região + histograma + proporção de claro/escuro.

Extraio a Rep2 separadamente em treino e teste. Como o cálculo é feito por imagem (não usa estatística do
conjunto), não há risco de vazamento.

In [ ]:
# Representacao 1: pixels brutos (ja estao em Xtr_d / Xte_d)
Xtr_d_rep1, Xte_d_rep1 = Xtr_d, Xte_d           # Rep1 sao os proprios pixels, sem transformacao

# Representacao 2: atributos derivados
Xtr_d_rep2 = extract_features(Xtr_d, DIGITS_SIDE, DIGITS_MAX, grid=4, n_bins=16)  # extrai a Rep2 do treino
Xte_d_rep2 = extract_features(Xte_d, DIGITS_SIDE, DIGITS_MAX, grid=4, n_bins=16)  # extrai a Rep2 do teste (mesmos parametros)

print("Rep1 (pixels)  - dimensao:", Xtr_d_rep1.shape[1], "features")   # numero de atributos da Rep1 (64)
print("Rep2 (derivada) - dimensao:", Xtr_d_rep2.shape[1], "features")  # numero de atributos da Rep2 (50)

## A.6 Treinamento

Rodo os três modelos nas duas representações. Cada chamada faz o GridSearchCV com a validação cruzada e depois
mede o desempenho no teste.

In [ ]:
print("Representacao 1 (pixels brutos):")       # cabecalho da Rep1
rows_d_rep1 = run_all_models(Xtr_d_rep1, Xte_d_rep1, ytr_d, yte_d, "Rep1-pixels", cv, SEED)  # roda os 3 modelos na Rep1

print("\nRepresentacao 2 (atributos derivados):")  # cabecalho da Rep2
rows_d_rep2 = run_all_models(Xtr_d_rep2, Xte_d_rep2, ytr_d, yte_d, "Rep2-derivada", cv, SEED)  # roda os 3 modelos na Rep2

results_digits = rows_d_rep1 + rows_d_rep2      # junta os 6 resultados (3 modelos x 2 representacoes)

## A.7 Avaliação — matriz de confusão e relatório

In [ ]:
# matrizes de confusao dos 3 modelos na Rep1
plot_confusions(rows_d_rep1, yte_d, DIGITS_NAMES,                    # plota as 3 matrizes de confusao (Rep1)
                "Digits - Matrizes de confusao (Rep1: pixels brutos)")

# Relatorio detalhado da melhor configuracao geral
best_d = best_config(results_digits)            # escolhe a melhor das 6 configuracoes (maior F1)
print(f"Melhor configuracao (Digits): {best_d['model']} | {best_d['representation']}")  # mostra qual venceu
print(f"Hiperparametros: {best_d['best_params']}\n")                  # mostra os hiperparametros vencedores
print(classification_report(yte_d, best_d["y_pred"], target_names=DIGITS_NAMES))  # precision/recall/F1 por classe

## A.8 Comparação (Digits)

Comparo os modelos entre si, as duas representações e o tempo de cada um.

In [ ]:
df_digits = results_to_df(results_digits)       # transforma os resultados do Digits numa tabela
display(df_digits.sort_values("test_f1", ascending=False).round(4)   # ordena pelo F1 (maior primeiro) e arredonda
        .reset_index(drop=True))                # reindexa a tabela do zero

plot_model_comparison(df_digits, "Digits - F1-macro por modelo e representacao")  # grafico de F1: modelo x representacao
plot_cost_comparison(df_digits, "Digits - Custo computacional")                   # grafico de tempos de treino/inferencia

**O que os números do Digits mostram**

Quem foi melhor foi o KNN com pixels brutos (k = 3): acurácia 0,967 e F1-macro 0,966, com a Random Forest logo
atrás (0,964 / 0,963). Três coisas chamaram atenção:

Primeiro, os pixels brutos ganharam da representação derivada em todos os modelos. No KNN a queda foi grande
(de 0,966 para 0,897 ao trocar pixels por atributos derivados): numa imagem 8×8 já centrada, os 64 pixels
praticamente já contam a história toda, e o histograma acaba perdendo a posição. A Random Forest sofreu menos
com a Rep2 (0,963 → 0,947), porque combina atributos em vez de medir distância no espaço inteiro.

Segundo, o Naive Bayes foi o pior dos três (acc ~0,82 nas duas representações). Faz sentido: pixels vizinhos
são bem correlacionados, e isso quebra a suposição de independência do NB. No relatório, os dígitos 8 (recall
0,89) e 9 (0,92) foram os mais difíceis, justamente os de traço mais ambíguo.

Terceiro, no custo dá para ver a teoria na prática: o KNN quase não tem treino (é lazy), mas a inferência é a
mais cara (~0,034 s contra ~0,002 s do NB); a Random Forest é o contrário, treino mais alto (~3 s) e
inferência rápida. O Naive Bayes é o mais barato dos dois lados.

---
# Parte B — CIFAR-10 (em tons de cinza)

O CIFAR-10 é um dos datasets mais conhecidos de visão computacional: 60.000 imagens coloridas 32×32 de 10
classes de objetos (avião, automóvel, pássaro, gato, veado, cachorro, sapo, cavalo, navio e caminhão). Como o
enunciado permite, converti tudo para tons de cinza (1 canal, 1.024 atributos) e usei só ML clássico.

Diferente do Digits, no CIFAR a cor, a textura e o fundo importam bastante. Tirando a cor e jogando pixels num
modelo clássico, eu já esperava uma acurácia bem mais baixa — e é justamente isso que torna o dataset
interessante para discutir até onde o ML clássico consegue ir em imagens.

## B.1 Carregamento

O carregador procura os arquivos em `/kaggle/input`, reconhece o formato Python oficial do CIFAR-10
(`data_batch_1..5` e `test_batch`) ou um `.npz`, e converte de RGB para cinza pela luminância
(0.299·R + 0.587·G + 0.114·B).

Para garantir a reprodutibilidade (rodar do início ao fim **sem ajustes do professor**), o carregador tem um
*fallback automático*: se não encontrar os dados anexados, ele baixa o CIFAR-10 da fonte oficial
(`TRY_DOWNLOAD = True`). Assim o notebook funciona tanto no Kaggle (offline, com o dataset anexado) quanto em
qualquer ambiente com internet.

- **No Kaggle (offline):** adicione pela aba *Datasets* do "+ Add Input" um dataset com a pasta
  `cifar-10-batches-py` (por exemplo `pankrzysiu/cifar10-python`). Tem que ser um *Dataset*, não um Notebook.
  Nesse caso o download nem é necessário.
- **Fora do Kaggle (ou sem dataset anexado):** basta ter internet ligada que o download acontece sozinho.

In [ ]:
def _cifar_unpickle(path):
    """Le um batch do CIFAR-10 no formato Python (pickle)."""
    import pickle                              # modulo para ler o formato pickle do CIFAR
    with open(path, "rb") as f:                # abre o arquivo em modo binario
        return pickle.load(f, encoding="bytes")  # carrega o dicionario do batch (chaves em bytes)


def _rgb_to_gray(X_rgb):
    """Converte CIFAR RGB (N, 3072) -> tons de cinza (N, 1024).

    O CIFAR armazena cada imagem como 1024 valores R, 1024 G e 1024 B (nessa ordem).
    Usamos a luminancia padrao: 0.299 R + 0.587 G + 0.114 B.
    """
    r = X_rgb[:, 0:1024]                       # primeiros 1024 valores = canal vermelho (R)
    g = X_rgb[:, 1024:2048]                    # proximos 1024 = canal verde (G)
    b = X_rgb[:, 2048:3072]                    # ultimos 1024 = canal azul (B)
    return (0.299 * r + 0.587 * g + 0.114 * b).astype(np.float32)  # combina os canais na luminancia (cinza)


# Fallback automatico: o carregador tenta os dados locais (Kaggle) e, se nao achar,
# baixa o CIFAR-10 da fonte oficial. Assim o notebook roda do inicio ao fim em
# qualquer ambiente (basta haver internet quando nao houver dataset anexado).
TRY_DOWNLOAD = True                            # True = baixa automaticamente se nao houver dados locais


def load_cifar10(try_download=TRY_DOWNLOAD):
    """Carrega o CIFAR-10 (em tons de cinza) procurando arquivos em /kaggle/input.

    Detecta automaticamente:
      A) formato Python oficial (arquivos data_batch_1..5 e test_batch);
      B) arquivos .npz / .npy (X com 3072 colunas RGB ou shape N,32,32,3).
    Converte tudo para cinza (N, 1024). Se nao houver dados locais e try_download=True,
    baixa o dataset da fonte oficial (Toronto) e usa esses arquivos.
    """
    import os, re, glob, socket                # os/re/glob: navegar e filtrar arquivos; socket: timeout do download

    root = "/kaggle/input"                     # pasta onde o Kaggle monta os datasets anexados
    all_files = []                             # vai acumular todos os caminhos de arquivo encontrados
    if os.path.isdir(root):                    # se a pasta do Kaggle existir...
        for dp, _, fns in os.walk(root):       # ...percorre recursivamente todas as subpastas
            for fn in fns:                     # para cada arquivo encontrado
                all_files.append(os.path.join(dp, fn))  # guarda o caminho completo

    print("Arquivos encontrados em /kaggle/input (ate 60):")  # mostra o que foi achado (ajuda a depurar)
    for f in all_files[:60]:                   # lista no maximo 60 arquivos
        print("   ", f)
    if not all_files:                          # se nada foi encontrado localmente
        print("   (nenhum) -- nenhum dataset anexado; vou tentar baixar automaticamente.")

    # --- A) Formato Python oficial: data_batch_1..5 e test_batch ---
    batches = sorted([f for f in all_files     # filtra os arquivos cujo nome e data_batch_N ou test_batch
                      if re.search(r"(data_batch_\d+|test_batch)$", os.path.basename(f))])
    if batches:                                # se achou os batches localmente
        Xs, ys = [], []                        # listas para acumular imagens e rotulos de cada batch
        for b in batches:                      # para cada arquivo de batch
            d = _cifar_unpickle(b)             # le o batch
            Xs.append(np.asarray(d[b"data"], dtype=np.float32))   # acumula os pixels (RGB)
            ys.append(np.asarray(d[b"labels"], dtype=int))        # acumula os rotulos
        X_rgb = np.vstack(Xs)                  # empilha todos os pixels num so array
        y = np.concatenate(ys)                 # concatena todos os rotulos
        X = _rgb_to_gray(X_rgb)                # converte RGB -> cinza
        print("OK: CIFAR-10 (formato Python). RGB", X_rgb.shape, "-> cinza", X.shape)
        return X, y                            # devolve imagens em cinza e rotulos

    # --- B) .npz / .npy ---
    for f in all_files:                        # procura algum arquivo .npz entre os encontrados
        fl = f.lower()
        if fl.endswith(".npz"):                # se for um .npz
            z = np.load(f)                     # abre o arquivo compactado
            keys = list(z.keys())              # lista as chaves de dentro dele
            xk = next((k for k in keys if "x" in k.lower() or "image" in k.lower()), keys[0])   # adivinha a chave das imagens
            yk = next((k for k in keys if "y" in k.lower() or "label" in k.lower()), keys[-1])  # adivinha a chave dos rotulos
            arr = z[xk]                        # pega o array de imagens
            y = z[yk].astype(int).ravel()      # pega os rotulos como vetor de inteiros
            arr = arr.reshape(arr.shape[0], -1).astype(np.float32)  # achata cada imagem num vetor
            X = _rgb_to_gray(arr) if arr.shape[1] == 3072 else arr  # converte para cinza so se vier em RGB (3072 cols)
            print("OK: CIFAR-10 (.npz). Shape:", X.shape)
            return X, y                        # devolve os dados

    # --- C) Download (fallback automatico) ---
    if try_download:                           # se permitido baixar (padrao)
        import tarfile, urllib.request, tempfile  # tarfile: descompactar; urllib: baixar; tempfile: pasta temporaria
        print("Sem dados locais; baixando CIFAR-10 da fonte oficial (precisa de internet)...")
        socket.setdefaulttimeout(120)          # limita a espera do download a 120s
        try:
            tmp = tempfile.mkdtemp()           # cria uma pasta temporaria
            tgz = os.path.join(tmp, "cifar.tar.gz")  # caminho do arquivo baixado
            urllib.request.urlretrieve("https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz", tgz)  # baixa o CIFAR
            with tarfile.open(tgz) as t:       # abre o .tar.gz
                t.extractall(tmp)              # extrai tudo na pasta temporaria
            dl = sorted(glob.glob(os.path.join(tmp, "**", "data_batch_*"), recursive=True) +   # acha os batches baixados...
                        glob.glob(os.path.join(tmp, "**", "test_batch"), recursive=True))      # ...e o test_batch
            Xs, ys = [], []                    # acumuladores (igual ao caso A)
            for b in dl:                       # para cada batch baixado
                d = _cifar_unpickle(b)         # le o batch
                Xs.append(np.asarray(d[b"data"], dtype=np.float32))   # acumula pixels
                ys.append(np.asarray(d[b"labels"], dtype=int))        # acumula rotulos
            X = _rgb_to_gray(np.vstack(Xs))    # empilha e converte para cinza
            y = np.concatenate(ys)             # concatena os rotulos
            print("OK: CIFAR-10 baixado. Shape (cinza):", X.shape)
            return X, y                        # devolve os dados baixados
        except Exception as e:                 # se o download falhar (sem internet, etc.)
            print("Download falhou:", repr(e)[:120])  # mostra o motivo resumido

    raise RuntimeError(                        # so chega aqui se nao achou local e o download falhou/desligado
        "Nenhum dado do CIFAR-10 foi encontrado em /kaggle/input e o download falhou.\n"
        "Opcao 1 (Kaggle, offline): Painel direito -> '+ Add Input' -> aba 'Datasets';\n"
        "  pesquise 'cifar-10 python' e adicione um dataset com os arquivos\n"
        "  data_batch_1..5 e test_batch (ex.: 'pankrzysiu/cifar10-python').\n"
        "Opcao 2 (qualquer ambiente): ligue a internet para o download automatico funcionar.\n"
        "Depois rode tudo de novo (Run All)."
    )


X_cifar_full, y_cifar_full = load_cifar10()    # carrega o CIFAR-10 completo em tons de cinza
print("\nShape completo (cinza):", X_cifar_full.shape)  # confere o shape: (60000, 1024)


## B.2 Subamostragem

O CIFAR tem 60 mil imagens e, com 1.024 atributos, rodar KNN e Random Forest com GridSearchCV em cima de tudo
fica caro (o KNN ainda tem inferência proporcional ao tamanho do treino). Por isso peguei uma subamostra
estratificada de `N_SUBSET` imagens, mantendo a proporção das classes. Dá para aumentar esse número se sobrar
tempo de máquina.

In [ ]:
# Nomes das 10 classes do CIFAR-10 (ordem oficial dos rotulos 0..9)
CIFAR_NAMES = ["aviao", "automovel", "passaro", "gato", "veado",      # rotulo 0..4
               "cachorro", "sapo", "cavalo", "navio", "caminhao"]     # rotulo 5..9
CIFAR_SIDE = 32   # imagens 32x32 convertidas para 1 canal (cinza) -> 1024 atributos

# Subamostra estratificada para viabilizar o custo computacional
N_SUBSET = 8000   # quantidade de imagens usadas (ajuste conforme o tempo disponivel)
if N_SUBSET < len(y_cifar_full):               # se a subamostra for menor que o total...
    X_cifar, _, y_cifar, _ = train_test_split(  # usa o train_test_split so para amostrar de forma estratificada
        X_cifar_full, y_cifar_full,
        train_size=N_SUBSET, stratify=y_cifar_full, random_state=SEED)  # mantem a proporcao das classes e fixa a semente
else:
    X_cifar, y_cifar = X_cifar_full, y_cifar_full  # se N_SUBSET >= total, usa tudo

CIFAR_MAX = float(X_cifar.max())   # intensidade maxima (~255), usada para normalizar na Rep2
print("Subconjunto de trabalho:", X_cifar.shape)   # confere o shape da subamostra
document_dataset("CIFAR-10 cinza (subconjunto)", X_cifar, y_cifar,  # imprime os metadados do subconjunto
                 CIFAR_SIDE, 1, CIFAR_NAMES)

## B.3 Visualização exploratória

In [ ]:
plot_class_examples(X_cifar, y_cifar, CIFAR_SIDE, CIFAR_NAMES,        # mostra um exemplo de cada classe do CIFAR
                    "CIFAR-10 - um exemplo por classe")
plot_class_distribution(y_cifar, CIFAR_NAMES,                        # mostra a distribuicao das classes na subamostra
                        "CIFAR-10 - distribuicao de classes")

O CIFAR também é balanceado (6.000 imagens por classe no total). Só que, ao contrário do Digits, as
imagens têm fundo, iluminação e poses variadas — e, sem a cor, fica difícil separar coisas como um sapo
(verde) de um automóvel. Já dá para imaginar bastante variação dentro de cada classe e confusões entre os
animais (gato, cachorro, veado, pássaro) e entre os veículos (automóvel, caminhão).

## B.4 Split e as duas representações

Mesma receita da Parte A: holdout estratificado 80/20 e as duas representações, com a escala continuando
dentro do Pipeline de cada modelo.

In [ ]:
# Holdout estratificado
Xtr_f, Xte_f, ytr_f, yte_f = train_test_split(                       # divide a subamostra em treino e teste
    X_cifar, y_cifar, test_size=0.20, stratify=y_cifar, random_state=SEED)  # 80/20, estratificado, semente fixa

# Representacao 1: pixels brutos (1024 features)
Xtr_f_rep1, Xte_f_rep1 = Xtr_f, Xte_f          # Rep1 = os 1024 pixels em cinza, sem transformacao

# Representacao 2: atributos derivados
Xtr_f_rep2 = extract_features(Xtr_f, CIFAR_SIDE, CIFAR_MAX, grid=4, n_bins=16)  # extrai a Rep2 do treino
Xte_f_rep2 = extract_features(Xte_f, CIFAR_SIDE, CIFAR_MAX, grid=4, n_bins=16)  # extrai a Rep2 do teste

print("Treino:", Xtr_f.shape, "| Teste:", Xte_f.shape)             # confere os tamanhos
print("Rep1 (pixels)  - dimensao:", Xtr_f_rep1.shape[1], "features")   # numero de atributos da Rep1 (1024)
print("Rep2 (derivada) - dimensao:", Xtr_f_rep2.shape[1], "features")  # numero de atributos da Rep2 (50)

## B.5 Treinamento

Esta é a parte mais demorada. Com `N_SUBSET = 8000`, conte alguns minutos — o gargalo é a Random Forest sobre
os 1.024 pixels.

In [ ]:
print("Representacao 1 (pixels brutos):")       # cabecalho da Rep1
rows_f_rep1 = run_all_models(Xtr_f_rep1, Xte_f_rep1, ytr_f, yte_f, "Rep1-pixels", cv, SEED)  # roda os 3 modelos na Rep1

print("\nRepresentacao 2 (atributos derivados):")  # cabecalho da Rep2
rows_f_rep2 = run_all_models(Xtr_f_rep2, Xte_f_rep2, ytr_f, yte_f, "Rep2-derivada", cv, SEED)  # roda os 3 modelos na Rep2

results_cifar = rows_f_rep1 + rows_f_rep2       # junta os 6 resultados do CIFAR

## B.6 Avaliação — matriz de confusão e relatório

In [ ]:
# Matrizes de confusao dos 3 modelos na Representacao 1
plot_confusions(rows_f_rep1, yte_f, CIFAR_NAMES,                     # plota as 3 matrizes de confusao (Rep1)
                "CIFAR-10 - Matrizes de confusao (Rep1: pixels brutos)")

# Relatorio da melhor configuracao
best_f = best_config(results_cifar)             # escolhe a melhor das 6 configuracoes do CIFAR (maior F1)
print(f"Melhor configuracao (CIFAR-10): {best_f['model']} | {best_f['representation']}")  # mostra qual venceu
print(f"Hiperparametros: {best_f['best_params']}\n")                  # mostra os hiperparametros vencedores
print(classification_report(yte_f, best_f["y_pred"], target_names=CIFAR_NAMES))  # precision/recall/F1 por classe

## B.7 Comparação experimental (CIFAR-10)


In [ ]:
df_cifar = results_to_df(results_cifar)         # transforma os resultados do CIFAR numa tabela
display(df_cifar.sort_values("test_f1", ascending=False).round(4)    # ordena pelo F1 (maior primeiro) e arredonda
        .reset_index(drop=True))                # reindexa a tabela do zero

plot_model_comparison(df_cifar, "CIFAR-10 - F1-macro por modelo e representacao")  # grafico de F1: modelo x representacao
plot_cost_comparison(df_cifar, "CIFAR-10 - Custo computacional")                   # grafico de tempos de treino/inferencia

**O que os números do CIFAR mostram**

A melhor combinação foi a Random Forest com atributos derivados (200 árvores): acurácia 0,380 e F1-macro
0,376. O ranking completo no teste ficou assim:

| Modelo | Representação | F1-macro | Acurácia |
|---|---|---|---|
| Random Forest | derivada | 0,376 | 0,380 |
| Random Forest | pixels | 0,366 | 0,369 |
| KNN | derivada | 0,312 | 0,319 |
| Naive Bayes | derivada | 0,256 | 0,273 |
| Naive Bayes | pixels | 0,222 | 0,244 |
| KNN | pixels | 0,201 | 0,222 |

A primeira coisa que salta aos olhos é a queda em relação ao Digits (~0,38 contra 0,97). Era esperado: sem cor
e com pixels ou descritores simples, o modelo clássico não capta forma, textura nem contexto. Ainda assim,
0,38 é quase quatro vezes o acaso (0,10), mas deixa claro o limite desse tipo de abordagem em imagens naturais.

O ponto mais interessante inverteu o que aconteceu no Digits: aqui a representação derivada ganhou dos pixels
em todos os modelos, e no KNN a diferença foi enorme (0,312 contra 0,201). Faz sentido — em 1.024 pixels de
cenas naturais, com o objeto em posições e fundos diferentes, os pixels viram quase ruído, e o KNN sofre
demais com a alta dimensionalidade. As estatísticas por região e o histograma resumem a imagem e acabam dando
atributos mais úteis. É aquela ideia de que a escolha das features pesa mais do que a do modelo.

Olhando a matriz de confusão da Random Forest (pixels), os erros são bem "humanos" e ficam em dois grupos. Nos
animais, o gato é o pior caso — acerta só 30 das 160 imagens — e se confunde com cachorro (27), veado (19) e
cavalo (18); sapo e pássaro também escorregam para veado. Nos veículos, automóvel vira caminhão (37), avião
vira navio (35), e caminhão se divide entre automóvel e navio. No relatório do melhor modelo isso aparece como
o pior recall sendo o do gato (0,20). Sem a cor, silhuetas parecidas em cinza (felino e cachorro, carro e
caminhão) ficam quase iguais para o modelo.

Por último o custo: treinar a Random Forest nos pixels foi de longe o mais pesado (~162 s, contra ~33 s na
representação derivada). Ou seja, a Rep2 deu F1 maior e ainda treinou umas 5 vezes mais rápido. O KNN nos
pixels tem a inferência mais lenta (~0,34 s), pela natureza lazy em alta dimensão, e o Naive Bayes é o mais
barato.

---
# 5. Discussão e conclusão

Aqui junto os resultados das duas partes para fechar o item 9 do enunciado: o compromisso entre desempenho e
simplicidade, os principais erros e as limitações do ML clássico em imagens.

In [ ]:
df_all = pd.concat([df_digits.assign(dataset="Digits"),              # junta as tabelas dos dois datasets...
                    df_cifar.assign(dataset="CIFAR-10")], ignore_index=True)  # ...marcando a qual dataset cada linha pertence
display(df_all.sort_values(["dataset", "test_f1"], ascending=[True, False])   # ordena por dataset e, dentro dele, por F1
        .round(4).reset_index(drop=True))       # arredonda e reindexa para exibir

### 5.1 Desempenho x simplicidade

Resumo dos melhores resultados (F1-macro no teste):

| Dataset | Melhor modelo | F1 | Pior configuração | F1 |
|---|---|---|---|---|
| Digits (8×8) | KNN + pixels (k=3) | 0,966 | Naive Bayes + pixels | 0,825 |
| CIFAR-10 cinza (32×32) | Random Forest + derivada | 0,376 | KNN + pixels | 0,201 |

E como cada modelo se comportou no geral:

| Modelo | Tipo | Treino | Inferência | Resumo |
|---|---|---|---|---|
| KNN | lazy | quase nada | caro (~0,34 s no CIFAR-pixels) | ganhou no Digits, mas afundou no CIFAR-pixels (0,20) |
| Random Forest | ensemble (bagging) | alto (~162 s no CIFAR-pixels) | rápido | o melhor nos dois, aguenta bem a alta dimensão |
| Naive Bayes | probabilístico | muito rápido | muito rápido | o mais fraco; pixels correlacionados quebram a suposição de independência |

A conclusão é que não existe um "melhor modelo" fixo: quem ganha muda com o problema e até com a
representação. A queda de F1 de 0,966 para 0,376 mostra bem o salto de dificuldade entre dígitos pequenos e
centrados e fotos naturais em cinza. No fim, é o velho compromisso entre ajustar bem e manter o modelo simples.

### 5.2 Principais erros
- Digits: as confusões ficam entre dígitos de traço parecido — 8 (recall 0,89) e 9 (0,92) foram os piores.
- CIFAR-10: o gato é o caso mais difícil (recall 0,20; acerta 30/160 na Random Forest), confundido com
  cachorro (27), veado (19) e cavalo (18). Os veículos também se misturam (automóvel → caminhão 37, avião →
  navio 35). São erros que fazem sentido visualmente: sem cor, silhuetas parecidas em cinza ficam quase iguais.

### 5.3 Pixels x atributos derivados
O resultado foi oposto nos dois datasets:
- No Digits (8×8, centrado), os pixels venceram (KNN 0,966 contra 0,897).
- No CIFAR (32×32, natural), a representação derivada venceu (RF 0,376 contra 0,366; KNN 0,312 contra 0,201) e
  ainda treinou cerca de 5 vezes mais rápido (33 s contra 162 s).

Ou seja, a melhor representação depende do dado. Em imagem pequena e centrada, os pixels já bastam; em cena
natural ruidosa e de alta dimensão, resumir a imagem ajuda e ainda barateia o treino.

### 5.4 Limitações do ML clássico
O teto de ~0,38 no CIFAR vem de dois lados: pixels e descritores manuais não capturam forma, textura nem
contexto, e o tom de cinza joga fora a cor (que separaria, por exemplo, um sapo de um automóvel). Ainda tem a
dimensionalidade: com 1.024 atributos, o KNN degrada bastante. É justamente esse buraco que as CNNs
preencheriam, aprendendo as features sozinhas — mas isso já está fora do escopo do trabalho.

### 5.5 Conclusão
No Digits eu ficaria com o KNN (k=3): simples e com F1 0,966. No CIFAR cinza, com a Random Forest sobre
atributos derivados, que deu o melhor F1 (0,376) e ainda foi bem mais rápida do que sobre pixels. Como as bases
são balanceadas, usei o F1-macro como métrica principal, que também não deixa a classe fraca (o gato) se
perder na média. A limitação principal é clara: ML clássico com imagem em cinza não vai longe em fotos
naturais. Como próximo passo, eu testaria trazer a cor de volta (ou um histograma de cor), um descritor de
bordas tipo HOG ou um PCA antes do KNN, para ver se as confusões diminuem.

---
### Referências
As decisões se apoiam nas aulas da disciplina: aprendizado supervisionado e seleção de modelo (aulas 3 e 6),
Naive Bayes (7), KNN (8), bagging e Random Forest (10), métricas de classificação (11), imagem digital (13) e
pré-processamento e data leakage (15).